In [20]:
#IMPORTS

import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
# from itertools import combinations
from scipy.optimize import curve_fit
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [21]:
#LOADING CONSTANTS

FPS = 120
LEN_ERR_MM = 0.5
GEOM = json.load(open("geometry.json"))

In [ ]:
def body(arm):
    mm, g = 1e-3, 1e-3 #unit conversion (mm, g -> m, kg)
    L, W = arm["length"], arm["width"]

    out = dict(stickers={})

    if "sticker_centre" in arm:
        name = "blue" if "mass_pivot" in arm else "red"
        out["stickers"][name] = arm["sticker_centre"]*mm

    if "joint" in arm:
        out["joint"] = arm["joint"]*mm

    parts = [(arm["mass"]*g, (L/2-arm["pivot_top"])*mm, arm["mass"]*g*((L*mm)**2+(W*mm)**2)/12)]

    if "mass_pivot" in arm:
        parts.append((arm["mass_pivot"]*g, 0, arm["mass_pivot"]*g*(arm["size_pivot"]*mm)**2/6))

    if "mass_joint" in arm:
        j = out["joint"]
        parts.append((arm["mass_joint"]*g, j, arm["mass_joint"]*g*(arm["diameter_joint"]*mm/2)**2/2))

    for x in arm.get("extras", []):
        parts.append((x["mass"]*g, x["along"]*mm, 0))

    m = sum(p[0] for p in parts)
    ca = sum(p[0]*p[1] for p in parts)/m
    Ip = sum(p[2]+p[0]*p[1]**2 for p in parts)

    out.update(m=m, com=ca, I_pivot=Ip, I_cm=Ip-m*ca**2)
    return out

ARM = {k: body(GEOM[k]) for k in ("arm1", "arm2")}
BLUE = ARM["arm1"]["stickers"]["blue"]
RED = ARM["arm2"]["stickers"]["red"]
JOINT = ARM["arm1"]["joint"]

{'arm1': {'stickers': {'blue': 0.20500000000000002}, 'joint': 0.20700000000000002, 'm': 0.6045999999999999, 'com': 0.09774429374793252, 'I_pivot': 0.009456190983333331, 'I_cm': 0.003679874651146762}, 'arm2': {'stickers': {'red': 0.14267949192431123}, 'm': 0.19469999999999998, 'com': 0.07, 'I_pivot': 0.00150568, 'I_cm': 0.0005516499999999999}}


In [9]:
DATA_DIR = Path("data")
runs = []
for f in sorted(DATA_DIR.glob("*.csv")):
    data = np.genfromtxt(f, dtype=float, delimiter=",", skip_header=1, missing_values="", filling_values=np.nan)
    name = f.stem.replace("_data", "")
    parts = name.split("_")

    runs.append(dict(
        name=name,
        init=int(parts[1]),
        run=int(parts[2]),
        t=data[:, 1],
        x_red=1920-data[:, 2],
        y_red=1080-data[:, 3],
        x_blue=1920-data[:, 4],
        y_blue=1080-data[:, 5],
        x_hinge=np.nanmedian(1920-data[:, 6]),
        y_hinge=np.nanmedian(1080-data[:, 7])
    ))

print(len(runs), "runs loaded")

23 runs loaded


In [ ]:
runs.append()

BLUE["along"]
JOINT["along"]
x_com = x_pivot + ARM["arm1"]["com"]["along"] * np.cos(theta)
y_com = y_pivot + ARM["arm1"]["com"]["along"] * np.sin(theta)

In [10]:
def fit_circle(x, y):
    """Returns (cx, cy, r), their errors, residuals in px (NaN where input is NaN), angle of each point."""
    ok = np.isfinite(x) & np.isfinite(y)
    xm, ym = x[ok], y[ok]
    f = lambda xy, cx, cy, r: np.hypot(xy[0] - cx, xy[1] - cy) - r
    p0 = [xm.mean(), ym.mean(), np.hypot(xm - xm.mean(), ym - ym.mean()).mean()]
    p, cov = curve_fit(f, (xm, ym), np.zeros_like(xm), p0=p0)
    res = np.full(len(x), np.nan)
    res[ok] = f((xm, ym), *p)
    return p, np.sqrt(np.diag(cov)), res, np.arctan2(y - p[1], x - p[0])

def coverage(phi):
    return np.mean(np.histogram(phi[np.isfinite(phi)], bins=36, range=(-np.pi, np.pi))[0] > 0)

def harmonics(phi, res, n=4):
    """residual = sum of cos/sin(k*angle). k=1 centre error, k=2 ellipse (tilt), k>=3 lens/tracking."""
    ok = np.isfinite(res)
    A = np.column_stack([np.ones(ok.sum())] + [fn(k * phi[ok]) for k in range(1, n + 1) for fn in (np.cos, np.sin)])
    coef = np.linalg.lstsq(A, res[ok], rcond=None)[0]
    model = np.full(len(res), np.nan)
    model[ok] = A @ coef
    return model, np.array([np.hypot(coef[2 * k - 1], coef[2 * k]) for k in range(1, n + 1)])

def harmonic_report(runs, key, title, verbose=True):
    phi = np.concatenate([r[f"phi_{key}"] for r in runs])
    res = np.concatenate([r[f"res_{key}"] for r in runs])
    model, amps = harmonics(phi, res)
    ok = np.isfinite(res)
    noise = np.std((res - model)[ok])
    if verbose:
        print(f"{title}: harmonics k=1..4 = {amps.round(3)} px (rough 1-sigma {noise*np.sqrt(2/ok.sum()):.3f}); random noise left = {noise:.3f} px")
    return phi, res, model, noise

def plot_bias(ax, runs, key, title):
    phi, res, model, noise = harmonic_report(runs, key, title)
    edges = np.linspace(-np.pi, np.pi, 37)
    idx = np.digitize(phi, edges) - 1
    mid = np.degrees((edges[:-1] + edges[1:]) / 2)
    m = np.array([np.nanmean(res[idx == i]) if np.any(idx == i) else np.nan for i in range(36)])
    s = np.array([np.nanstd(res[idx == i]) / np.sqrt(max((idx == i).sum(), 1)) for i in range(36)])
    ax.errorbar(mid, m, s, fmt="o", ms=3)
    ok = np.isfinite(model); o = np.argsort(phi[ok])
    ax.plot(np.degrees(phi[ok][o]), model[ok][o], "r-")
    ax.axhline(0, color="k", lw=0.5)
    ax.set(title=title, xlabel="angle around fitted centre (deg)", ylabel="radial residual (px)")

def fit_ellipse(x, y):
    """Rotated ellipse. q = minor/major axis ratio; tilt = arccos(q); alpha = direction of the major axis."""
    ok = np.isfinite(x) & np.isfinite(y)
    xm, ym = x[ok], y[ok]
    c0 = np.array([xm.mean(), ym.mean()])
    w, V = np.linalg.eigh(np.cov(np.vstack([xm - c0[0], ym - c0[1]])))
    alpha0 = np.arctan2(V[1, 1], V[0, 1])
    q0 = float(np.clip(np.sqrt(w[0] / w[1]), 0.8, 0.999))
    def f(xy, cx, cy, a, q, al):
        dx, dy = xy[0] - cx, xy[1] - cy
        return np.hypot(dx * np.cos(al) + dy * np.sin(al), (-dx * np.sin(al) + dy * np.cos(al)) / q) - a
    p, cov = curve_fit(f, (xm, ym), np.zeros_like(xm), p0=[*c0, np.sqrt(2 * w[1]), q0, alpha0])
    err = np.sqrt(np.diag(cov))
    cx, cy, a, q, al = p
    if q > 1:
        a, q, al = a * q, 1 / q, al + np.pi / 2
    al = (al + np.pi / 2) % np.pi - np.pi / 2
    return dict(c=(cx, cy), q=q, alpha=al, alpha_deg=np.degrees(al), tilt_deg=np.degrees(np.arccos(min(q, 1.0))),
                tilt_err_deg=np.degrees(err[3] / max(np.sqrt(1 - q**2), 1e-6)))

def tilt_matrix(q, alpha):
    c, s = np.cos(alpha), np.sin(alpha)
    R = np.array([[c, -s], [s, c]])
    return R @ np.diag([1, 1 / q]) @ R.T

def smooth(y, n=5):
    return np.convolve(np.pad(y, (n // 2, n - 1 - n // 2), mode="edge"), np.ones(n) / n, mode="valid")